# Deliverable 1 (EDA) - Matilde: behaviour, value and feature engineering
Group 27 - Data Mining 2026/27 - draft (`notebooks/drafts/eda_matilde.ipynb`)

**Scope (from the work plan):** usage, add-ons, value, engineered features, first look at possible groups.

**Conventions shared with the group:** `df_original` (raw copy), `df_raw`, `df_clean`, snake_case features,
`metric_features` / `non_metric_features` lists.

> **Before running:** put `acdc_dataset.csv` (Moodle) in `DATA_PATH` below. When Martim shares `clean_data()`,
> switch `USE_CLEAN_DATA = True` in the setup section.

## Setup

In [ ]:
# Remember: library imports are ALWAYS at the top of the script, no exceptions!
import os
from math import ceil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# for better resolution plots
%config InlineBackend.figure_format = 'retina'

sns.set_theme(style="whitegrid")

DATA_PATH = os.path.join("..", "..", "data", "acdc_dataset.csv")   # <- adjust if needed
USE_CLEAN_DATA = False   # True once Martim's clean_data() is available

## Reading the data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_original = df_raw.copy()   # always keep a copy of the raw data

print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

### Metadata (Annex 8.2 of the guidelines)
Dates: `membership_start`, `membership_end`, `last_badge_swipe`. Usage: `total_visits`, `avg_weekly_visits`,
`days_since_last_visit`, `entitled_visits_last_period`, `actual_visits_last_period`, `meeting_room_hours`,
`event_space_hours`, `guest_passes_issued`. Add-on flags: `uses_*`. Value: `n_renewals`, `lifetime_value`.

### Data preparation shared by all parts

In [ ]:
# Dates (not in labs: pd.to_datetime)
date_cols = ["membership_start", "membership_end", "last_badge_swipe"]
df = df_raw.copy()
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# uses_* flags can come as 0/1, True/False or Yes/No -> make them 0/1 integers
addon_cols = [c for c in df.columns if c.startswith("uses_")]
truthy = {"1", "true", "yes", "y", "t"}
for c in addon_cols:
    df[c] = (df[c].astype(str).str.strip().str.lower().isin(truthy)).astype(int)

print("add-on flags:", addon_cols)

In [ ]:
# Extraction date = last_badge_swipe + days_since_last_visit (should be the same for every member)
extraction_candidates = (df["last_badge_swipe"] + pd.to_timedelta(df["days_since_last_visit"], unit="D"))
print(extraction_candidates.value_counts().head())

# CONFIRM with the group that this is the real extraction date
extraction_date = extraction_candidates.mode().iloc[0]
print("Extraction date used:", extraction_date.date())

In [ ]:
if USE_CLEAN_DATA:
    from clean_data import clean_data   # Martim's function
    df_clean = clean_data(df_raw)
    for c in date_cols:
        df_clean[c] = pd.to_datetime(df_clean[c], errors="coerce")
else:
    df_clean = df.copy()   # placeholder until clean_data() exists

non_metric_features = ["gender", "country", "industry", "plan_tier", "is_team_account"] + addon_cols
metric_features = ["age", "n_colleagues_on_account", "plan_days_per_week", "total_visits", "avg_weekly_visits",
                   "days_since_last_visit", "entitled_visits_last_period", "actual_visits_last_period",
                   "meeting_room_hours", "event_space_hours", "guest_passes_issued", "n_renewals", "lifetime_value"]
metric_features = [c for c in metric_features if c in df_clean.columns]

# 1. Usage

### Distributions of visits and recency
Histograms and box plots for each usage variable (same subplot-loop habit as lab04).

In [ ]:
usage_features = ["total_visits", "avg_weekly_visits", "days_since_last_visit",
                  "entitled_visits_last_period", "actual_visits_last_period"]

fig, axes = plt.subplots(2, len(usage_features), figsize=(20, 7), tight_layout=True)
for i, feat in enumerate(usage_features):
    sns.histplot(df_clean[feat], bins=30, ax=axes[0, i])
    axes[0, i].set_title(feat)
    sns.boxplot(x=df_clean[feat], ax=axes[1, i])
fig.suptitle("Usage variables: histograms and box plots")
plt.show()

In [ ]:
df_clean[usage_features].describe().round(2).T

### Last period: actual vs entitled visits

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(df_clean["entitled_visits_last_period"], df_clean["actual_visits_last_period"],
               gridsize=25, cmap="viridis", mincnt=1)
lim = max(df_clean["entitled_visits_last_period"].max(), df_clean["actual_visits_last_period"].max())
ax.plot([0, lim], [0, lim], color="red", linestyle="--", label="actual = entitled")
ax.set_xlabel("entitled_visits_last_period")
ax.set_ylabel("actual_visits_last_period")
ax.set_title("Actual vs entitled visits (last period)")
ax.legend()
fig.colorbar(hb, ax=ax, label="members")
plt.show()

print("Share of members who exceeded their entitlement:",
      round((df_clean["actual_visits_last_period"] > df_clean["entitled_visits_last_period"]).mean() * 100, 2), "%")

### Insights (usage)
- *To fill in after running:* typical visit frequency, skewness, share of inactive members (high recency),
  share above/below entitlement.

# 2. Add-ons

In [ ]:
adoption = df_clean[addon_cols].mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=adoption.values, y=adoption.index, ax=ax, color="tab:blue")
ax.set_xlabel("% of members who use it")
ax.set_title("Add-on adoption rate")
for p, v in zip(ax.patches, adoption.values):
    ax.text(v + 0.5, p.get_y() + p.get_height() / 2, f"{v:.1f}%", va="center")
plt.show()

### Which add-ons are used together?

In [ ]:
# Co-usage matrix: share of members using both add-ons (rows/cols = add-ons)
co_use = pd.DataFrame(index=addon_cols, columns=addon_cols, dtype=float)
for a in addon_cols:
    for b in addon_cols:
        co_use.loc[a, b] = ((df_clean[a] == 1) & (df_clean[b] == 1)).mean() * 100

plt.figure(figsize=(10, 8))
sns.heatmap(co_use.round(1), annot=True, fmt=".1f", cmap="Blues", square=True, linewidths=.5)
plt.title("% of members using both add-ons")
plt.show()

In [ ]:
# Correlation between the binary flags (phi coefficient = Pearson on 0/1)
corr_addons = df_clean[addon_cols].corr().round(2)
mask = np.triu(np.ones_like(corr_addons, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr_addons, mask=mask, center=0, cmap="PiYG", vmin=-1, vmax=1,
            annot=np.where(np.abs(corr_addons) >= 0.2, corr_addons.values, ""),
            fmt="s", square=True, linewidths=.5)
plt.title("Correlation between add-on flags (|r| >= 0.2 annotated)")
plt.show()

### Meeting-room and event-space hours

In [ ]:
hours_features = ["meeting_room_hours", "event_space_hours"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), tight_layout=True)
for i, feat in enumerate(hours_features):
    sns.histplot(df_clean[feat], bins=30, ax=axes[0, i])
    axes[0, i].set_title(feat)
    sns.boxplot(x=df_clean[feat], ax=axes[1, i])
plt.show()

print("Share of members with zero hours:")
print((df_clean[hours_features] == 0).mean().round(3))

### KPI row

In [ ]:
kpi = {
    "members": len(df_clean),
    "avg lifetime_value": df_clean["lifetime_value"].mean(),
    "avg weekly visits": df_clean["avg_weekly_visits"].mean(),
    "avg # add-ons": df_clean[addon_cols].sum(axis=1).mean(),
}
pd.Series(kpi).round(2).to_frame("value")

### Insights (add-ons)
- *To fill in after running:* most/least adopted add-ons, pairs that go together, share of members with no add-on.

# 3. Value

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), tight_layout=True)
sns.histplot(df_clean["lifetime_value"], bins=40, ax=axes[0, 0]); axes[0, 0].set_title("lifetime_value")
sns.boxplot(x=df_clean["lifetime_value"], ax=axes[1, 0])
sns.countplot(x=df_clean["n_renewals"], ax=axes[0, 1]); axes[0, 1].set_title("n_renewals")
sns.boxplot(x=df_clean["n_renewals"], ax=axes[1, 1])
plt.show()

### What drives lifetime value?

In [ ]:
drivers = [c for c in metric_features if c != "lifetime_value"]
corr_ltv = df_clean[drivers + ["lifetime_value"]].corr()["lifetime_value"].drop("lifetime_value").sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=corr_ltv.values, y=corr_ltv.index, ax=ax, color="tab:green")
ax.set_xlabel("Pearson correlation with lifetime_value")
ax.set_title("Linear association with LTV")
plt.show()

In [ ]:
# LTV by plan tier (categorical vs numeric, lab05 style)
tier_order = ["hot_desk", "flex", "dedicated_desk", "full_time", "private_office"]
tier_order = [t for t in tier_order if t in df_clean["plan_tier"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
sns.boxplot(data=df_clean, x="plan_tier", y="lifetime_value", order=tier_order, ax=axes[0])
axes[0].set_title("lifetime_value by plan_tier")
axes[0].tick_params(axis="x", rotation=25)
sns.boxplot(data=df_clean, x="plan_tier", y="n_renewals", order=tier_order, ax=axes[1])
axes[1].set_title("n_renewals by plan_tier")
axes[1].tick_params(axis="x", rotation=25)
plt.show()

df_clean.groupby("plan_tier")["lifetime_value"].agg(["count", "mean", "median"]).round(1).reindex(tier_order)

### Insights (value)
- *To fill in after running:* skewness of LTV, concentration of value (e.g. share of LTV from the top 10% of members),
  whether plan tier or tenure explains most of the LTV.

In [ ]:
# Concentration of value
ltv_sorted = df_clean["lifetime_value"].sort_values(ascending=False)
top10 = ltv_sorted.head(int(len(ltv_sorted) * 0.10)).sum() / ltv_sorted.sum()
print(f"Top 10% of members generate {top10:.1%} of total lifetime value")

# 4. Feature engineering

| Feature | Formula | What it captures |
|---|---|---|
| `tenure_days` | (`membership_end` or extraction date) - `membership_start` | Loyalty |
| `utilization_rate` | actual / entitled visits (last period) | Value obtained from the plan; churn risk |
| `ltv_per_month` | `lifetime_value` / tenure in months | Worth per month, regardless of tenure |
| `n_addons` | sum of `uses_*` | How much of the service they use |
| `hours_per_visit` | (meeting + event hours) / `total_visits` | Office vs just a desk |
| `is_active` | `membership_end` is missing | Active vs churned |
| `plan_tier_ord` | hot_desk=1 ... private_office=5 | Plan tier as a numeric scale |
| `r_score`, `f_score`, `m_score`, `rfm_score` | quantiles of recency, frequency, LTV | Value-based segmentation (RFM) |

In [ ]:
df_fe = df_clean.copy()

# --- tenure
end_or_extraction = df_fe["membership_end"].fillna(extraction_date)
df_fe["tenure_days"] = (end_or_extraction - df_fe["membership_start"]).dt.days
# a tenure <= 0 is a data problem (flag it, do not silently fix)
print("Members with tenure_days <= 0:", (df_fe["tenure_days"] <= 0).sum())

# --- ratios (avoid inf when the denominator is 0)
df_fe["utilization_rate"] = (df_fe["actual_visits_last_period"]
                             / df_fe["entitled_visits_last_period"].replace(0, np.nan))
tenure_months = (df_fe["tenure_days"] / 30.44).where(df_fe["tenure_days"] > 0)
df_fe["ltv_per_month"] = df_fe["lifetime_value"] / tenure_months
df_fe["hours_per_visit"] = ((df_fe["meeting_room_hours"] + df_fe["event_space_hours"])
                            / df_fe["total_visits"].replace(0, np.nan))

# --- counts / flags
df_fe["n_addons"] = df_fe[addon_cols].sum(axis=1)
df_fe["is_active"] = df_fe["membership_end"].isna().astype(int)

# --- ordinal plan tier
tier_map = {"hot_desk": 1, "flex": 2, "dedicated_desk": 3, "full_time": 4, "private_office": 5}
df_fe["plan_tier_ord"] = df_fe["plan_tier"].map(tier_map)
print("plan_tier values not mapped:", df_fe.loc[df_fe["plan_tier_ord"].isna(), "plan_tier"].unique())

In [ ]:
# --- RFM scores (1 = worst, 5 = best)
# Recency: FEWER days since last visit is better, so the scale is reversed
df_fe["r_score"] = pd.qcut(df_fe["days_since_last_visit"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
df_fe["f_score"] = pd.qcut(df_fe["avg_weekly_visits"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_fe["m_score"] = pd.qcut(df_fe["lifetime_value"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df_fe["rfm_score"] = df_fe[["r_score", "f_score", "m_score"]].sum(axis=1)

engineered_features = ["tenure_days", "utilization_rate", "ltv_per_month", "n_addons", "hours_per_visit",
                       "is_active", "plan_tier_ord", "r_score", "f_score", "m_score", "rfm_score"]
df_fe[engineered_features].describe().round(2).T

### Quality check of the engineered features
Infinities and NaNs created by the ratios, and extreme values that will need capping (coordinate with Martim's outlier work).

In [ ]:
check = pd.DataFrame({
    "n_nan": df_fe[engineered_features].isna().sum(),
    "n_inf": np.isinf(df_fe[engineered_features]).sum(),
    "min": df_fe[engineered_features].min(),
    "max": df_fe[engineered_features].max(),
})
check

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), tight_layout=True)
for ax, feat in zip(axes.flatten(), ["tenure_days", "utilization_rate", "ltv_per_month", "n_addons",
                                      "hours_per_visit", "plan_tier_ord", "rfm_score", "is_active"]):
    sns.histplot(df_fe[feat].replace([np.inf, -np.inf], np.nan).dropna(), bins=30, ax=ax)
    ax.set_title(feat)
fig.suptitle("Engineered features: distributions")
plt.show()

# 5. First look at possible groups (analysis only, no clustering yet)
Pair plots of the engineered features coloured by `plan_tier` and by active/churned.

In [ ]:
pair_features = ["tenure_days", "utilization_rate", "ltv_per_month", "n_addons", "hours_per_visit"]
df_pair = df_fe[pair_features + ["plan_tier", "is_active"]].replace([np.inf, -np.inf], np.nan).dropna()
df_pair["status"] = df_pair["is_active"].map({1: "active", 0: "churned"})

# 2000-row sample keeps the plot fast; this is only a visual check
sample = df_pair.sample(min(2000, len(df_pair)), random_state=42)

In [ ]:
g = sns.pairplot(sample, vars=pair_features, hue="plan_tier", hue_order=tier_order,
                 diag_kind="hist", plot_kws={"alpha": 0.4, "s": 12})
g.figure.suptitle("Engineered features by plan tier", y=1.02, fontsize=16)
plt.show()

In [ ]:
g = sns.pairplot(sample, vars=pair_features, hue="status",
                 diag_kind="hist", plot_kws={"alpha": 0.4, "s": 12})
g.figure.suptitle("Engineered features: active vs churned", y=1.02, fontsize=16)
plt.show()

In [ ]:
# Correlation among engineered features (candidates for redundancy, relevant for Deliverable 2)
corr_fe = df_fe[engineered_features].replace([np.inf, -np.inf], np.nan).corr().round(2)
plt.figure(figsize=(11, 9))
sns.heatmap(corr_fe, center=0, cmap="PiYG", vmin=-1, vmax=1,
            annot=np.where(np.abs(corr_fe) >= 0.5, corr_fe.values, ""), fmt="s",
            square=True, linewidths=.5)
plt.title("Engineered features: correlation (|r| >= 0.5 annotated)")
plt.show()

### Insights for slides 3 and 4 (fill in after running with the real data)
- Which engineered features separate the groups best?
- Natural groupings suggested by the plots and the expected number of clusters (with the evidence).
- Challenges anticipated (skewness, outliers, redundancy such as `total_visits` vs `avg_weekly_visits`, imbalanced tiers).